In [4]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pooch
import scanpy as sc
import scvi
import torch

In [5]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.1.5


In [6]:
sc.set_figure_params(figsize=(4, 4), frameon=False)
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor" : "w"}
%config InlineBackend.figure_format="retina"

In [7]:
adata = sc.read("../process/framework/obj/combined_E12_E14_signac.h5ad")

In [16]:
adata = sc.read("../process/framework/obj/combined_E12_E14_signc_sceasy.h5ad")

In [19]:
adata.X[1:10,1:10]

<Compressed Sparse Column sparse matrix of dtype 'float64'
	with 0 stored elements and shape (9, 9)>

In [26]:
adata.obs["batch"]

B1_AAACGAAAGAATACTG-1    B1
B1_AAACGAAAGACGACTG-1    B1
B1_AAACGAAAGAGAGTTT-1    B1
B1_AAACGAAAGCCTTTGA-1    B1
B1_AAACGAAAGCGTGTTT-1    B1
                         ..
C2_TTTGTGTGTAGCGAGT-1    C2
C2_TTTGTGTGTCCTATTT-1    C2
C2_TTTGTGTTCCATCTAT-1    C2
C2_TTTGTGTTCCGTGCAG-1    C2
C2_TTTGTGTTCGTTGTAG-1    C2
Name: batch, Length: 42042, dtype: category
Categories (4, object): ['B1', 'B2', 'C1', 'C2']

In [20]:
print("# regions before filtering:", adata.shape[-1])

# compute the threshold: 5% of the cells
min_cells = int(adata.shape[0] * 0.05)
# in-place filtering of regions
sc.pp.filter_genes(adata, min_cells=min_cells)

print("# regions after filtering:", adata.shape[-1])

# regions before filtering: 297386
# regions after filtering: 32923


In [21]:
adata.X[1:10,1:10].toarray()

array([[0., 1., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [24]:
scvi.model.PEAKVI.setup_anndata(adata,batch_key = "batch")

/home/hanlitian/miniforge3/envs/py310/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:116: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


In [ ]:
model = scvi.model.PEAKVI(adata)
model.train()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/hanlitian/miniforge3/envs/py310/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.
/home/hanlitian/miniforge3/envs/py310/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.


Epoch 93/500:  18%|█▊        | 92/500 [8:29:58<36:43:41, 324.07s/it, v_num=1, train_loss_step=6.7e+7, train_loss_epoch=1.66e+8] 